# Aegis Wave - Ultra-Robust Smoke Test
1. **Sanitizes Data**: Replaces NaNs/Infs with 0.
2. **Dense Model Test**: Checks if simple math works on real data.
3. **Conv1D Model Test**: Checks if the Convolutional kernel is the culprit.

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import json
import tensorflow as tf

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.config.set_visible_devices([], "GPU")

DATA_DIR = "data/"
SUB_COUNT = 16
TIME_STEPS = 100
SAMPLE_LIMIT = 50

In [ ]:
print("Step 1: Loading & Sanitizing Data...")
X_list, y_list = [], []
metadata_df = pd.read_csv(os.path.join(DATA_DIR, "metadata/sample_metadata.csv"))
with open(os.path.join(DATA_DIR, "splits/train_id.json"), "r") as f:
    ids = json.load(f)[:SAMPLE_LIMIT]

for sample_id in ids:
    row = metadata_df[metadata_df["id"] == sample_id]
    path = os.path.join(DATA_DIR, row.iloc[0]["file_path"].lstrip("./"))
    try:
        with h5py.File(path, "r") as f:
            d = f["CSI_amps"][:].squeeze()[:64:4, :TIME_STEPS].T
            # remove nans/infs
            d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
            X_list.append(d.astype("float32"))
            y_list.append(0 if row.iloc[0]["label"] == "Fall" else 1)
    except: continue

X_train = np.array(X_list)
y_train = np.array(y_list)
print(f"Loaded {len(X_train)} sanitized samples.")
print(f"Data Max: {np.max(X_train)}, Min: {np.min(X_train)}")

In [ ]:
print("Step 2: Testing Dense Model...")
model_dense = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TIME_STEPS, SUB_COUNT)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(2, activation="softmax")
])
model_dense.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model_dense.fit(X_train, y_train, epochs=2, verbose=1)
print("Dense Model Passed")

In [ ]:
print("Step 3: Testing Conv1D Model...")
model_conv = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TIME_STEPS, SUB_COUNT)),
    tf.keras.layers.Conv1D(8, 3, activation="relu"),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(2, activation="softmax")
])
model_conv.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model_conv.fit(X_train, y_train, epochs=2, verbose=1)
print("Conv1D Model Passed")